# Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

  
if Path.cwd().name in ['ETL', 'data_science']:
    root = Path.cwd().parent
else:
    root = Path.cwd()

if str(root) not in sys.path:
    sys.path.append(str(root))
else:
    None
from utils import align_working_dir, save_chart

align_working_dir('data_science') 

%run ./0_content_load.ipynb

In [ ]:
# Save charts to a specific folder with a given prefix in the name.
CHART_PREFIX = "article_"

# Annual Thematic Evolution by Categories
- **Stacked Bar Chart**  

In [ ]:
all_years = sorted(combined_df['Year'].unique())

pivot_df = (
    combined_df[combined_df['Type'] == 'Article']
    .groupby(['Year', 'Category'])
    .size()
    .unstack(fill_value=0)
)

# Reindex to include missing years
# This forces the DataFrame to include all years, filling missing rows with 0
pivot_df = pivot_df.reindex(all_years, fill_value=0)

# Normalize to 100% 
# We use .fillna(0) because dividing 0 by 0 (empty years) results in NaN
pivot_pct = pivot_df.div(pivot_df.sum(axis=1), axis=0).fillna(0) * 100

category_totals = pivot_df.sum()  # Total count per category
total_articles = int(category_totals.sum())  # Grand total of all articles

# --- Plotting ---
ax = pivot_pct.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6),
    colormap='tab10',
    width=0.8,
    rot=0
)

fig = ax.get_figure()

fig.suptitle('Articles: Annual Thematic Evolution')
ax.set_title(f'n={total_articles}')

ax.set_ylabel('Percentage of Total Articles (%)')
ax.set_xlabel('Year')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Prepare the legend labels with counts
# This creates a list like: ['Category A (n=150)', 'Category B (n=200)', ...]
legend_labels = [f'{col} (n={int(category_totals[col])})' for col in pivot_pct.columns]

# Apply the custom legend labels
ax.legend(
    legend_labels, 
    title='Article Categories:', 
    bbox_to_anchor=(1.02, 1), 
    loc='upper left',
    frameon=False,
    alignment='left'
)

# Add percentage labels inside the bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5: # Only label bars large enough
        x, y = p.get_xy()
        ax.annotate(f'{height:.0f}%', (x + width/2, y + height/2), ha='center', va='center',
fontsize=16, color='white', fontweight='medium')

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_thematic_evolution", CHART_PREFIX)
plt.show()

# Publishing Routine (Day of the Week)
-  **Bar Chart**  

Does the organization have a specific day they prefer to publish articles? This reveals their operational habit.

In [ ]:
pub_routine_df = combined_df[combined_df['Type'] == 'Article'].copy()

# Extract Day of Week (Same as before)
pub_routine_df['Day_of_Week'] = combined_df['Date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 5))

sns.countplot(
    data=pub_routine_df,
    x='Day_of_Week',
    order=day_order,
    color=CUSTOM_PALETTE["Article"]  
)

fig.suptitle('Publishing Routine')
ax.set_title('Which day are Articles released?\n Period: 2019-2025', linespacing=1.5)

ax.set_ylabel('Number of Articles')
ax.set_xlabel('Days of the Week')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "publishing_routine", CHART_PREFIX)
plt.show()

# Top 10 Most Frequent Article Tags
- **Horizontal Bar Chart**  

In [ ]:

# "Explode" the tags list so each tag gets its own row
# Add to a new cell
all_tags = articles_df.query("publishedDate.dt.year != 2026").copy()
all_tags = all_tags.explode('tags')
top_tags = all_tags['tags'].value_counts().head(10).reset_index()
top_tags.columns = ['Tag', 'Count']

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
       data=top_tags,
       y='Tag',
       x='Count',
    color=CUSTOM_PALETTE["Article"]  
   )

sns.barplot(data=top_tags, y='Tag', x='Count', color=CUSTOM_PALETTE["Article"]  )
fig.suptitle('Top 10 Most Frequent Article Tags')
ax.set_title("Period: 2019-2025")

ax.set_xlabel('Occurrences')
ax.set_ylabel('')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "top_tags", CHART_PREFIX)
plt.show()

# Editorial Depth (Word Count Distribution)
- **Distribution Chart**   

* **Logic**: Calculate the word count of the contentText.
* **Insight**: Are we writing short "snippets" or long-form "treatises"? How has our "depth" changed
     over the years?

In [ ]:
art_word_count_df = articles_df.query("publishedDate.dt.year != 2026").copy()

# Create a numerical column for word count
# We split the text by whitespace and count the resulting list
art_word_count_df['word_count'] = art_word_count_df['contentText'].apply(lambda x: len(str(x).split()))

# Calculate stats on the new numerical column
mean_val = art_word_count_df['word_count'].mean()
median_val = art_word_count_df['word_count'].median()

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

# Make sure to plot the 'word_count' column, not 'contentText'
ax = sns.histplot(art_word_count_df['word_count'], bins=15, kde=True, color=CUSTOM_PALETTE["Article"])

plt.axvline(mean_val, color='red', linestyle='--', label=f'Mean (Avg): {mean_val:.0f}')
plt.axvline(median_val, color='green', linestyle='-', label=f'Median (Middle): {median_val:.0f}')

fig.suptitle('Distribution of Article Length')
ax.set_title('Mean vs Median Depth Analysis\n Period: 2019-2025', linespacing=1.5)

ax.set_xlabel('Estimated Word Count')
ax.set_ylabel('Number of Articles')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

ax.legend()

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "length", CHART_PREFIX)
plt.show()

# Annual Article Activity Pulse
- **Heatmap**  

In [ ]:
# Filter for events and create the pivot table
article_pulse_df = (
       combined_df.query("Type == 'Article'")
       .groupby(['Year', 'Month'])
       .size()
       .unstack(fill_value=0)
)

# Define the FULL range of years and months you want to see
all_years = range(2019, combined_df['Year'].max() + 1) # Start from 2019
all_months = range(1, 13)

# 3. Use .reindex to force the DataFrame to include all years and months
# This automatically fills any "missing" year/month with 0
article_pulse_df = article_pulse_df.reindex(index=all_years, columns=all_months,
   fill_value=0)

# Ensure all 12 months are represented even if no events happened
for month in range(1, 13):
       if month not in article_pulse_df.columns:
           article_pulse_df[month] = 0

# REVERSE the order of the years (rows) 
# This puts the most recent year (2025) at the top
article_pulse_df = article_pulse_df.iloc[::-1]

# Sort months correctly (1-12)
article_pulse_df = article_pulse_df.reindex(columns=range(1, 13))

# 3. Map month numbers to names for the x-axis
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
'Oct', 'Nov', 'Dec']
article_pulse_df.columns = month_names

# --- Plotting ---
fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(
       article_pulse_df,
       annot=True,
       fmt='d',
       cmap='BuPu',
       linewidths=.5,
       cbar_kws={'label': 'Number of Articles Published'},
       ax=ax,
       annot_kws={"size": 12, "weight": "bold"}
)

# Manually turn on ONLY the left and bottom spines
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)

for spine in ['left', 'bottom']:
    ax.spines[spine].set_color('#292929') 
    ax.spines[spine].set_linewidth(1.0)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

fig.suptitle('Annual Article Activity Pulse', x=0.08, y=0.98, ha='left')
ax.set_title(f'Frequency of articles published per month across the years | Total:{total_articles}', loc='left', pad=20)

ax.set_ylabel('Year')
ax.set_xlabel('Month of Year')

# Ensure the Y-axis years are horizontal and readable
plt.yticks(rotation=0)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_activity_pulse", CHART_PREFIX)
plt.show()